# Notebook 01 — Generación de Datos Sintéticos
**Objetivo:** Generar 1000 ejemplos de contratos usando Llama 3.1 70B como modelo teacher

**Entradas:** Token de Databricks (Colab Secrets), URL del workspace

**Salidas:** `/content/drive/MyDrive/agente-contratos-cto/dataset/contratos_sft.jsonl`

**Tiempo estimado:** ~30 minutos

In [ ]:
!pip install -q openai==1.40.0

In [ ]:
from google.colab import userdata, drive
from openai import OpenAI
import json, time, random
from pathlib import Path

# Montar Drive
drive.mount('/content/drive')

# Crear estructura de carpetas
RUTA_BASE = Path('/content/drive/MyDrive/agente-contratos-cto')
RUTA_DATASET = RUTA_BASE / 'dataset'
RUTA_DATASET.mkdir(parents=True, exist_ok=True)

# Verificar que los secrets estén configurados antes de continuar
assert userdata.get('DATABRICKS_TOKEN'), (
    "Agregar DATABRICKS_TOKEN en Colab Secrets "
    "(panel izquierdo → ícono 🔑 → Nuevo secreto)"
)
assert userdata.get('DATABRICKS_HOST'), (
    "Agregar DATABRICKS_HOST en Colab Secrets "
    "(ej: https://xxx.cloud.databricks.com)"
)

# Cliente Databricks — token desde Colab Secrets
DATABRICKS_HOST = userdata.get('DATABRICKS_HOST')
DATABRICKS_TOKEN = userdata.get('DATABRICKS_TOKEN')

cliente = OpenAI(
    api_key=DATABRICKS_TOKEN,
    base_url=f"{DATABRICKS_HOST}/serving-endpoints"
)

MODELO_TEACHER = "databricks-meta-llama-3-1-70b-instruct"
N_EJEMPLOS = 1000
TAMANO_LOTE = 10

## Definición de Tipos de Tarea y Plantillas de Prompts

In [ ]:
# ---------------------------------------------------------------------------
# Tipos de tarea que el agente de contratos debe dominar
# ---------------------------------------------------------------------------
TIPOS_TAREA: list[str] = [
    "analizar_clausula",
    "comparar_propuestas",
    "alerta_renovacion",
    "estrategia_negociacion",
]

# ---------------------------------------------------------------------------
# Proveedores y contexto realista Chile / Latam
# ---------------------------------------------------------------------------
PROVEEDORES: list[str] = [
    "AWS", "Microsoft Azure", "Google Cloud Platform",
    "SAP", "Oracle", "Salesforce",
    "IBM", "ServiceNow", "Datadog",
    "Snowflake", "MongoDB Atlas", "Confluent",
    "Cloudflare", "Twilio", "Stripe",
]

TIPOS_CONTRATO: list[str] = [
    "SaaS Enterprise", "IaaS con compromiso de consumo",
    "PaaS con soporte premium", "Licenciamiento perpetuo con mantenimiento",
    "Suscripci\u00f3n anual con descuento por volumen",
    "Contrato marco de servicios gestionados",
    "Acuerdo de nivel de servicio (SLA) dedicado",
    "Contrato de consultor\u00eda e implementaci\u00f3n",
    "Licencia por usuario con tier enterprise",
    "Contrato de soporte t\u00e9cnico 24/7",
]

DURACIONES: list[str] = [
    "12 meses", "24 meses", "36 meses",
    "6 meses (piloto)", "48 meses", "60 meses",
]

RANGOS_MONTO_USD: list[tuple[int, int]] = [
    (5_000, 25_000),
    (25_000, 100_000),
    (100_000, 500_000),
    (500_000, 2_000_000),
    (2_000_000, 10_000_000),
]

ESCENARIOS_CTO: list[str] = [
    "Migraci\u00f3n de infraestructura on-premise a la nube",
    "Renovaci\u00f3n de contrato existente con renegociaci\u00f3n de precios",
    "Evaluaci\u00f3n de nuevo proveedor para reemplazar soluci\u00f3n actual",
    "Escalamiento de plataforma por crecimiento de usuarios 3x",
    "Consolidaci\u00f3n de m\u00faltiples contratos en un acuerdo marco",
    "Negociaci\u00f3n post-incidente de SLA por ca\u00edda de servicio",
    "Adopci\u00f3n de nueva tecnolog\u00eda de IA/ML en producci\u00f3n",
    "Cumplimiento regulatorio de datos en Latinoam\u00e9rica",
    "Optimizaci\u00f3n de costos cloud tras auditor\u00eda de consumo",
    "Contrataci\u00f3n de servicios de ciberseguridad gestionada",
    "Implementaci\u00f3n de ERP para filial en Chile",
    "Expansi\u00f3n regional con requerimientos de residencia de datos",
]


def _contexto_aleatorio() -> dict[str, str]:
    """Genera un contexto aleatorio para enriquecer la diversidad de los prompts."""
    rango = random.choice(RANGOS_MONTO_USD)
    monto = random.randint(rango[0], rango[1])
    return {
        "proveedor": random.choice(PROVEEDORES),
        "proveedor_alternativo": random.choice(PROVEEDORES),
        "tipo_contrato": random.choice(TIPOS_CONTRATO),
        "duracion": random.choice(DURACIONES),
        "monto_usd": f"{monto:,}",
        "escenario": random.choice(ESCENARIOS_CTO),
    }


# ---------------------------------------------------------------------------
# Plantillas de prompt por tipo de tarea
# Cada plantilla instruye a Llama 70B a generar UN ejemplo diverso y realista
# ---------------------------------------------------------------------------
PLANTILLAS_PROMPT: dict[str, str] = {
    # ------------------------------------------------------------------
    "analizar_clausula": """Eres un experto legal en contratos tecnol\u00f3gicos para empresas en Chile y Latinoam\u00e9rica.
Genera UN ejemplo realista y detallado de an\u00e1lisis de cl\u00e1usula contractual.

Contexto:
- Proveedor: {proveedor}
- Tipo de contrato: {tipo_contrato}
- Duraci\u00f3n: {duracion}
- Monto aproximado: USD {monto_usd}
- Escenario: {escenario}

Debes generar un JSON con exactamente estos campos:
- "instruccion": una instrucci\u00f3n clara pidiendo analizar una cl\u00e1usula espec\u00edfica (limitaci\u00f3n de responsabilidad, penalidades, SLA, propiedad intelectual, confidencialidad, terminaci\u00f3n anticipada, etc.)
- "entrada": el texto completo de la cl\u00e1usula contractual (m\u00ednimo 80 palabras, con lenguaje legal realista, mencionando al proveedor y montos)
- "salida": un an\u00e1lisis profesional de la cl\u00e1usula que incluya: resumen ejecutivo, riesgos identificados, recomendaciones concretas para el CTO, y sugerencias de modificaci\u00f3n (m\u00ednimo 120 palabras)

IMPORTANTE:
- Usa lenguaje profesional en espa\u00f1ol
- Incluye montos, porcentajes y plazos espec\u00edficos
- Var\u00eda los tipos de cl\u00e1usula (no repitas siempre la misma)
- El an\u00e1lisis debe ser \u00fatil para un CTO real

Responde SOLO con el JSON, sin texto adicional.""",

    # ------------------------------------------------------------------
    "comparar_propuestas": """Eres un consultor senior de adquisiciones tecnol\u00f3gicas para empresas en Chile y Latinoam\u00e9rica.
Genera UN ejemplo realista y detallado de comparaci\u00f3n de propuestas de proveedores.

Contexto:
- Proveedor principal: {proveedor}
- Proveedor alternativo: {proveedor_alternativo}
- Tipo de contrato: {tipo_contrato}
- Duraci\u00f3n: {duracion}
- Presupuesto: USD {monto_usd}
- Escenario: {escenario}

Debes generar un JSON con exactamente estos campos:
- "instruccion": una instrucci\u00f3n clara pidiendo comparar dos o m\u00e1s propuestas de proveedores tecnol\u00f3gicos
- "entrada": descripci\u00f3n detallada de las propuestas a comparar, incluyendo precios, condiciones, SLAs, soporte, y caracter\u00edsticas diferenciadoras de cada proveedor (m\u00ednimo 100 palabras)
- "salida": una comparaci\u00f3n profesional con: tabla comparativa resumida (en texto), ventajas y desventajas de cada opci\u00f3n, an\u00e1lisis de costo total de propiedad (TCO), y recomendaci\u00f3n final justificada para el CTO (m\u00ednimo 150 palabras)

IMPORTANTE:
- Usa montos realistas en USD
- Incluye m\u00e9tricas comparables (uptime, tiempo de respuesta de soporte, etc.)
- Considera factores como residencia de datos en Latam
- La recomendaci\u00f3n debe ser matizada, no siempre el m\u00e1s barato

Responde SOLO con el JSON, sin texto adicional.""",

    # ------------------------------------------------------------------
    "alerta_renovacion": """Eres un gestor de contratos tecnol\u00f3gicos para una empresa en Chile y Latinoam\u00e9rica.
Genera UN ejemplo realista y detallado de alerta de renovaci\u00f3n de contrato.

Contexto:
- Proveedor: {proveedor}
- Tipo de contrato: {tipo_contrato}
- Duraci\u00f3n original: {duracion}
- Monto actual: USD {monto_usd}
- Escenario: {escenario}

Debes generar un JSON con exactamente estos campos:
- "instruccion": una instrucci\u00f3n clara pidiendo generar o evaluar una alerta de renovaci\u00f3n pr\u00f3xima
- "entrada": datos del contrato actual incluyendo: fecha de inicio, fecha de vencimiento, condiciones actuales, cl\u00e1usulas de renovaci\u00f3n autom\u00e1tica, penalidades por no renovar, y uso actual del servicio (m\u00ednimo 80 palabras)
- "salida": un informe de alerta que incluya: resumen de estado, d\u00edas restantes para actuar, riesgos de renovaci\u00f3n autom\u00e1tica, oportunidades de renegociaci\u00f3n, acciones recomendadas con cronograma, y estimaci\u00f3n de ahorro potencial (m\u00ednimo 120 palabras)

IMPORTANTE:
- Usa fechas realistas (2024-2026)
- Incluye plazos de notificaci\u00f3n (30, 60, 90 d\u00edas)
- Menciona riesgos de lock-in y cl\u00e1usulas de salida
- Sugiere palancas de negociaci\u00f3n concretas

Responde SOLO con el JSON, sin texto adicional.""",

    # ------------------------------------------------------------------
    "estrategia_negociacion": """Eres un estratega de negociaci\u00f3n de contratos tecnol\u00f3gicos con experiencia en Chile y Latinoam\u00e9rica.
Genera UN ejemplo realista y detallado de estrategia de negociaci\u00f3n.

Contexto:
- Proveedor: {proveedor}
- Tipo de contrato: {tipo_contrato}
- Duraci\u00f3n objetivo: {duracion}
- Monto en negociaci\u00f3n: USD {monto_usd}
- Escenario: {escenario}

Debes generar un JSON con exactamente estos campos:
- "instruccion": una instrucci\u00f3n clara pidiendo desarrollar una estrategia de negociaci\u00f3n para un contrato tecnol\u00f3gico
- "entrada": contexto completo de la negociaci\u00f3n incluyendo: situaci\u00f3n actual, necesidades del negocio, posici\u00f3n del proveedor, alternativas disponibles, presupuesto aprobado, y restricciones t\u00e9cnicas (m\u00ednimo 100 palabras)
- "salida": estrategia detallada con: objetivos de negociaci\u00f3n (BATNA, punto de reserva, aspiraci\u00f3n), t\u00e1cticas espec\u00edficas por fase, argumentos clave, concesiones aceptables vs. l\u00edneas rojas, cronograma de negociaci\u00f3n, y resultado esperado con m\u00e9tricas de \u00e9xito (m\u00ednimo 150 palabras)

IMPORTANTE:
- Incluye t\u00e1cticas de negociaci\u00f3n reales (anclaje, bundling, multi-year discount)
- Menciona benchmarks de mercado
- Considera la relaci\u00f3n a largo plazo con el proveedor
- Incluye escenarios de fallback

Responde SOLO con el JSON, sin texto adicional.""",
}

print(f"Tipos de tarea definidos: {len(TIPOS_TAREA)}")
print(f"Proveedores disponibles: {len(PROVEEDORES)}")
print(f"Escenarios CTO: {len(ESCENARIOS_CTO)}")
print(f"Combinaciones posibles de contexto: ~{len(PROVEEDORES) * len(TIPOS_CONTRATO) * len(DURACIONES) * len(RANGOS_MONTO_USD) * len(ESCENARIOS_CTO):,}")

## Función de Generación con Reintentos

In [ ]:
def generar_ejemplo(tipo_tarea: str, indice: int) -> dict | None:
    """Genera un ejemplo sint\u00e9tico de contrato usando Llama 3.1 70B.

    Realiza una llamada al modelo teacher a trav\u00e9s de la API de Databricks,
    aplicando reintentos con backoff exponencial en caso de fallos.

    Args:
        tipo_tarea: Uno de los tipos definidos en TIPOS_TAREA
                    ('analizar_clausula', 'comparar_propuestas',
                     'alerta_renovacion', 'estrategia_negociacion').
        indice: N\u00famero secuencial del ejemplo (para logging).

    Returns:
        Diccionario con las claves 'instruccion', 'entrada', 'salida' y
        'tipo_tarea', o None si la generaci\u00f3n falla tras todos los reintentos.
    """
    MAX_REINTENTOS: int = 3
    contexto: dict[str, str] = _contexto_aleatorio()
    prompt_formateado: str = PLANTILLAS_PROMPT[tipo_tarea].format(**contexto)

    for intento in range(1, MAX_REINTENTOS + 1):
        try:
            respuesta = cliente.chat.completions.create(
                model=MODELO_TEACHER,
                messages=[
                    {
                        "role": "system",
                        "content": (
                            "Eres un asistente experto en contratos tecnol\u00f3gicos. "
                            "Responde \u00fanicamente con JSON v\u00e1lido en espa\u00f1ol."
                        ),
                    },
                    {"role": "user", "content": prompt_formateado},
                ],
                temperature=0.9,
                max_tokens=2048,
            )

            # Limpiar respuesta de posibles bloques de c\u00f3digo markdown
            texto_crudo: str = respuesta.choices[0].message.content
            texto_limpio: str = (
                texto_crudo
                .replace("```json", "")
                .replace("```", "")
                .strip()
            )

            ejemplo: dict = json.loads(texto_limpio)

            # Validar que los campos requeridos est\u00e9n presentes
            campos_requeridos: list[str] = ["instruccion", "entrada", "salida"]
            for campo in campos_requeridos:
                if campo not in ejemplo:
                    raise KeyError(f"Campo requerido ausente: '{campo}'")

            # Agregar metadato de tipo de tarea
            ejemplo["tipo_tarea"] = tipo_tarea

            return ejemplo

        except json.JSONDecodeError as error_json:
            print(
                f"  [Intento {intento}/{MAX_REINTENTOS}] Error JSON en "
                f"ejemplo {indice} ({tipo_tarea}): {error_json}"
            )
        except KeyError as error_campo:
            print(
                f"  [Intento {intento}/{MAX_REINTENTOS}] Campo faltante en "
                f"ejemplo {indice} ({tipo_tarea}): {error_campo}"
            )
        except Exception as error_general:
            print(
                f"  [Intento {intento}/{MAX_REINTENTOS}] Error en "
                f"ejemplo {indice} ({tipo_tarea}): {error_general}"
            )

        # Backoff exponencial: 2s, 4s, 8s
        tiempo_espera: float = 2 ** intento
        print(f"  Esperando {tiempo_espera}s antes de reintentar...")
        time.sleep(tiempo_espera)

    print(f"  FALLO DEFINITIVO: ejemplo {indice} ({tipo_tarea}) tras {MAX_REINTENTOS} intentos.")
    return None


# Prueba r\u00e1pida con un ejemplo
print("Generando ejemplo de prueba...")
ejemplo_prueba: dict | None = generar_ejemplo("analizar_clausula", 0)
if ejemplo_prueba:
    print("Ejemplo generado exitosamente.")
    print(json.dumps(ejemplo_prueba, indent=2, ensure_ascii=False)[:500] + "...")
else:
    print("Error en la prueba. Verifica la conexi\u00f3n a Databricks.")

## Generación del Dataset Completo

In [ ]:
# ---------------------------------------------------------------------------
# Ruta del archivo de salida
# ---------------------------------------------------------------------------
ARCHIVO_SALIDA: Path = RUTA_DATASET / 'contratos_sft.jsonl'
ARCHIVO_PROGRESO: Path = RUTA_DATASET / 'progreso_generacion.json'
INTERVALO_GUARDADO: int = 50  # Guardar progreso cada N ejemplos

# ---------------------------------------------------------------------------
# Cargar progreso existente si la sesi\u00f3n se interrumpi\u00f3
# ---------------------------------------------------------------------------
ejemplos_generados: list[dict] = []
conteo_por_tipo: dict[str, int] = {t: 0 for t in TIPOS_TAREA}
indice_inicio: int = 0

if ARCHIVO_SALIDA.exists():
    print("Encontrado archivo de progreso previo. Cargando...")
    with open(ARCHIVO_SALIDA, 'r', encoding='utf-8') as archivo:
        for linea in archivo:
            linea = linea.strip()
            if linea:
                ejemplo = json.loads(linea)
                ejemplos_generados.append(ejemplo)
                tipo = ejemplo.get('tipo_tarea', 'desconocido')
                if tipo in conteo_por_tipo:
                    conteo_por_tipo[tipo] += 1
    indice_inicio = len(ejemplos_generados)
    print(f"Progreso cargado: {indice_inicio} ejemplos existentes.")
    print(f"Distribuci\u00f3n actual: {conteo_por_tipo}")
else:
    print("No se encontr\u00f3 progreso previo. Iniciando desde cero.")

# ---------------------------------------------------------------------------
# Construir la cola de tareas pendientes (round-robin: 250 por tipo)
# ---------------------------------------------------------------------------
EJEMPLOS_POR_TIPO: int = N_EJEMPLOS // len(TIPOS_TAREA)  # 250

cola_tareas: list[str] = []
for tipo in TIPOS_TAREA:
    pendientes: int = EJEMPLOS_POR_TIPO - conteo_por_tipo[tipo]
    if pendientes > 0:
        cola_tareas.extend([tipo] * pendientes)

# Mezclar para distribuir las llamadas de forma balanceada
random.shuffle(cola_tareas)

total_pendientes: int = len(cola_tareas)
print(f"\nEjemplos pendientes por generar: {total_pendientes}")
print(f"Objetivo por tipo: {EJEMPLOS_POR_TIPO}")

# ---------------------------------------------------------------------------
# Bucle principal de generaci\u00f3n
# ---------------------------------------------------------------------------
fallos: list[dict] = []  # Registro de fallos para an\u00e1lisis posterior
tiempo_inicio: float = time.time()

for i, tipo_tarea in enumerate(cola_tareas):
    indice_global: int = indice_inicio + i
    numero_ejemplo: int = indice_global + 1

    # Progreso cada TAMANO_LOTE ejemplos
    if i % TAMANO_LOTE == 0:
        transcurrido: float = time.time() - tiempo_inicio
        velocidad: str = f"{i / transcurrido:.1f} ej/s" if transcurrido > 0 and i > 0 else "calculando..."
        print(
            f"\n[{numero_ejemplo}/{N_EJEMPLOS}] "
            f"Generando... ({i}/{total_pendientes} pendientes) "
            f"| Velocidad: {velocidad}"
        )

    # Generar ejemplo
    ejemplo: dict | None = generar_ejemplo(tipo_tarea, numero_ejemplo)

    if ejemplo is not None:
        ejemplos_generados.append(ejemplo)
        conteo_por_tipo[tipo_tarea] += 1

        # Guardar al archivo JSONL de forma incremental
        with open(ARCHIVO_SALIDA, 'a', encoding='utf-8') as archivo:
            archivo.write(json.dumps(ejemplo, ensure_ascii=False) + '\n')
    else:
        fallos.append({
            "indice": numero_ejemplo,
            "tipo_tarea": tipo_tarea,
            "timestamp": time.strftime('%Y-%m-%d %H:%M:%S'),
        })

    # Guardar progreso y estad\u00edsticas cada INTERVALO_GUARDADO ejemplos
    if (i + 1) % INTERVALO_GUARDADO == 0:
        progreso: dict = {
            "total_generados": len(ejemplos_generados),
            "conteo_por_tipo": conteo_por_tipo,
            "total_fallos": len(fallos),
            "ultimo_guardado": time.strftime('%Y-%m-%d %H:%M:%S'),
        }
        with open(ARCHIVO_PROGRESO, 'w', encoding='utf-8') as archivo_prog:
            json.dump(progreso, archivo_prog, ensure_ascii=False, indent=2)
        print(
            f"  >> Progreso guardado en Drive: {len(ejemplos_generados)} ejemplos "
            f"| Fallos: {len(fallos)} | Distribuci\u00f3n: {conteo_por_tipo}"
        )

    # Pausa para evitar rate limiting
    time.sleep(0.5)

# ---------------------------------------------------------------------------
# Resumen final
# ---------------------------------------------------------------------------
tiempo_total: float = time.time() - tiempo_inicio
minutos: float = tiempo_total / 60

print("\n" + "=" * 60)
print("GENERACI\u00d3N COMPLETADA")
print("=" * 60)
print(f"Total generados: {len(ejemplos_generados)}")
print(f"Total fallos: {len(fallos)}")
print(f"Distribuci\u00f3n por tipo: {conteo_por_tipo}")
print(f"Tiempo total: {minutos:.1f} minutos")
print(f"Archivo guardado en: {ARCHIVO_SALIDA}")

if fallos:
    print(f"\nFallos registrados ({len(fallos)}):")
    for fallo in fallos[:10]:  # Mostrar m\u00e1ximo los primeros 10
        print(f"  - Ejemplo {fallo['indice']} ({fallo['tipo_tarea']})")

## Verificación y Estadísticas del Dataset

In [ ]:
def verificar_dataset(ruta_archivo: Path) -> None:
    """Verifica la integridad y muestra estad\u00edsticas del dataset generado.

    Carga el archivo JSONL, valida que cada registro contenga los campos
    requeridos, imprime la distribuci\u00f3n por tipo de tarea, y muestra
    ejemplos aleatorios para inspecci\u00f3n visual.

    Args:
        ruta_archivo: Ruta al archivo JSONL del dataset.
    """
    # Cargar todos los registros
    registros: list[dict] = []
    errores_linea: list[int] = []

    with open(ruta_archivo, 'r', encoding='utf-8') as archivo:
        for num_linea, linea in enumerate(archivo, start=1):
            linea = linea.strip()
            if not linea:
                continue
            try:
                registro: dict = json.loads(linea)
                registros.append(registro)
            except json.JSONDecodeError:
                errores_linea.append(num_linea)

    print("=" * 60)
    print("VERIFICACI\u00d3N DEL DATASET")
    print("=" * 60)
    print(f"Archivo: {ruta_archivo}")
    print(f"Total de registros: {len(registros)}")

    if errores_linea:
        print(f"L\u00edneas con errores de JSON: {errores_linea}")
    else:
        print("L\u00edneas con errores de JSON: 0")

    # Validar campos requeridos
    campos_requeridos: list[str] = ["instruccion", "entrada", "salida", "tipo_tarea"]
    registros_incompletos: list[int] = []

    for idx, registro in enumerate(registros):
        for campo in campos_requeridos:
            if campo not in registro or not registro[campo]:
                registros_incompletos.append(idx)
                break

    print(f"Registros con campos faltantes: {len(registros_incompletos)}")

    # Distribuci\u00f3n por tipo de tarea
    print("\n--- Distribuci\u00f3n por tipo de tarea ---")
    distribucion: dict[str, int] = {}
    for registro in registros:
        tipo: str = registro.get("tipo_tarea", "desconocido")
        distribucion[tipo] = distribucion.get(tipo, 0) + 1

    for tipo, cantidad in sorted(distribucion.items()):
        porcentaje: float = (cantidad / len(registros)) * 100 if registros else 0
        barra: str = "#" * int(porcentaje / 2)
        print(f"  {tipo:<30} {cantidad:>5} ({porcentaje:5.1f}%) {barra}")

    # Estad\u00edsticas de longitud
    print("\n--- Estad\u00edsticas de longitud (caracteres) ---")
    for campo in ["instruccion", "entrada", "salida"]:
        longitudes: list[int] = [
            len(r.get(campo, "")) for r in registros if r.get(campo)
        ]
        if longitudes:
            promedio: float = sum(longitudes) / len(longitudes)
            minimo: int = min(longitudes)
            maximo: int = max(longitudes)
            print(f"  {campo:<15} min={minimo:>5} | promedio={promedio:>7.0f} | max={maximo:>5}")

    # Mostrar 2 ejemplos aleatorios
    print("\n--- Ejemplos aleatorios ---")
    if len(registros) >= 2:
        muestra: list[dict] = random.sample(registros, 2)
    else:
        muestra = registros

    for i, ejemplo in enumerate(muestra, start=1):
        print(f"\n{'~' * 60}")
        print(f"Ejemplo {i} | Tipo: {ejemplo.get('tipo_tarea', 'N/A')}")
        print(f"{'~' * 60}")
        print(f"INSTRUCCI\u00d3N: {ejemplo.get('instruccion', 'N/A')[:200]}")
        print(f"ENTRADA:     {ejemplo.get('entrada', 'N/A')[:300]}...")
        print(f"SALIDA:      {ejemplo.get('salida', 'N/A')[:300]}...")

    print("\n" + "=" * 60)
    if not registros_incompletos and not errores_linea and len(registros) >= N_EJEMPLOS:
        print("RESULTADO: Dataset VALIDO y completo.")
    elif len(registros) > 0:
        print(
            f"RESULTADO: Dataset parcial ({len(registros)}/{N_EJEMPLOS}). "
            f"Revisar fallos si los hay."
        )
    else:
        print("RESULTADO: Dataset VACIO. Verificar generaci\u00f3n.")
    print("=" * 60)


# Ejecutar verificaci\u00f3n
verificar_dataset(ARCHIVO_SALIDA)